In [1]:
import os
import json
import re
import pandas as pd

pd.options.display.max_columns = None

In [2]:
def extract_json(response: str):
    """Extract JSON content from a formatted string."""
    match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if match:
        json_str = match.group(1)
    else:
        json_str = response.strip('```json').strip('```')
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Chyba při dekódování JSON: {e}")
        return None

In [3]:
def process_txt_files(folder_path, prefix):
    """Zpracuje všechny txt soubory začínající prefixem (např. 'bank_part') ve složce a vrátí Pandas DataFrame."""
    all_data = []
    
    # Get files with prefix and end with .txt
    files = [f for f in os.listdir(folder_path) if f.startswith(prefix) and f.endswith(".txt")]
    
    # Order by number
    files.sort(key=lambda x: int(re.search(r'chunk(\d+)', x).group(1)))
    
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as file:
            for line in file:
                try:
                    json_obj = json.loads(line.strip())
                    content_str = json_obj.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
                    extracted_json = extract_json(content_str)
                    
                    if extracted_json:
                        row = {"id": json_obj["id"], "custom_id": json_obj["custom_id"]}
                        for feature in extracted_json.get("features", []):
                            row[feature["feature_name"]] = feature["answer"]
                        
                        all_data.append(row)
                except json.JSONDecodeError:
                    print(f"Chyba dekódování JSON v souboru {filename}")

    df = pd.DataFrame(all_data)
    return df

In [4]:
# Použití skriptu
folder_path = "./"
df = process_txt_files(folder_path, "hate")

# Zobrazení výsledného dataframe
df

,id,custom_id,Presence of Racial Slurs,Mention of Ethnic Groups,Sentiment Polarity,Use of Violent Language,Presence of Hate Symbols,Use of Sarcasm,Text Length,Use of Expletives,Grammatical Errors,Use of First-Person Pronouns,Use of Second-Person Pronouns,Use of Third-Person Pronouns,Presence of Questions,Presence of Negations,Use of Hyperbolic Language,Presence of Emotive Language,Use of Metaphors,Presence of Direct Quotes,Use of Capitalization for Emphasis,Presence of Lists
0,batch_req_67c67638a93c8190bb71b26f520c368e,0,No,No,Neutral,No,No,No,Medium,No,No,No,No,Yes,No,No,No,No,No,No,No,No
1,batch_req_67c67638c37081909ae7b84b911fb321,1,No,No,Neutral,No,No,No,Medium,No,No,Yes,Yes,No,No,No,No,No,No,No,No,No
2,batch_req_67c67638d6d0819096c81e188903afc6,2,No,No,Neutral,No,No,No,Short,No,No,No,No,No,No,No,No,No,No,No,No,No
3,batch_req_67c67638eb008190a70be21f3e5b601d,3,No,Yes,Negative,Yes,No,No,Long,No,No,No,No,Yes,No,No,Yes,Yes,No,No,Yes,No
4,batch_req_67c67638fa70819095a705d14e5438b7,4,No,No,Neutral,No,No,No,Short,No,No,No,Yes,No,No,No,No,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6398,batch_req_67c6c33896848190b6db737a7828ca06,6398,No,No,Negative,Yes,No,No,Short,No,Yes,Yes,No,Yes,No,No,No,Yes,No,Yes,No,No
6399,batch_req_67c6c338a55c8190a996b747cee02850,6399,No,No,Negative,No,No,Yes,Short,No,No,No,No,Yes,No,Yes,No,Yes,No,No,No,No
6400,batch_req_67c6c338b5608190957ff59ba826b894,6400,No,No,Positive,No,No,No,Short,No,Yes,No,No,No,No,No,No,Yes,No,No,No,No
6401,batch_req_67c6c338c54481908af647f8f7593526,6401,No,No,Positive,No,No,No,Medium,No,No,No,No,No,No,No,No,Yes,No,No,No,No


# Merge with target

In [5]:
from datasets import load_dataset

data_load = load_dataset("odegiber/hate_speech18")

In [6]:
load_dataset = pd.DataFrame(data_load['train'])
#load_dataset_test = pd.DataFrame(data_load['test'])
#load_dataset = pd.concat([load_dataset, load_dataset_test], axis=0)
df['label'] = list(load_dataset['label'])[:len(df)]

In [7]:
df

,id,custom_id,Presence of Racial Slurs,Mention of Ethnic Groups,Sentiment Polarity,Use of Violent Language,Presence of Hate Symbols,Use of Sarcasm,Text Length,Use of Expletives,Grammatical Errors,Use of First-Person Pronouns,Use of Second-Person Pronouns,Use of Third-Person Pronouns,Presence of Questions,Presence of Negations,Use of Hyperbolic Language,Presence of Emotive Language,Use of Metaphors,Presence of Direct Quotes,Use of Capitalization for Emphasis,Presence of Lists,label
0,batch_req_67c67638a93c8190bb71b26f520c368e,0,No,No,Neutral,No,No,No,Medium,No,No,No,No,Yes,No,No,No,No,No,No,No,No,0
1,batch_req_67c67638c37081909ae7b84b911fb321,1,No,No,Neutral,No,No,No,Medium,No,No,Yes,Yes,No,No,No,No,No,No,No,No,No,0
2,batch_req_67c67638d6d0819096c81e188903afc6,2,No,No,Neutral,No,No,No,Short,No,No,No,No,No,No,No,No,No,No,No,No,No,0
3,batch_req_67c67638eb008190a70be21f3e5b601d,3,No,Yes,Negative,Yes,No,No,Long,No,No,No,No,Yes,No,No,Yes,Yes,No,No,Yes,No,1
4,batch_req_67c67638fa70819095a705d14e5438b7,4,No,No,Neutral,No,No,No,Short,No,No,No,Yes,No,No,No,No,No,No,No,No,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6398,batch_req_67c6c33896848190b6db737a7828ca06,6398,No,No,Negative,Yes,No,No,Short,No,Yes,Yes,No,Yes,No,No,No,Yes,No,Yes,No,No,1
6399,batch_req_67c6c338a55c8190a996b747cee02850,6399,No,No,Negative,No,No,Yes,Short,No,No,No,No,Yes,No,Yes,No,Yes,No,No,No,No,0
6400,batch_req_67c6c338b5608190957ff59ba826b894,6400,No,No,Positive,No,No,No,Short,No,Yes,No,No,No,No,No,No,Yes,No,No,No,No,0
6401,batch_req_67c6c338c54481908af647f8f7593526,6401,No,No,Positive,No,No,No,Medium,No,No,No,No,No,No,No,No,Yes,No,No,No,No,0


In [8]:
df = df.drop(columns=["id", "custom_id"])
data = pd.get_dummies( 
        df, sparse=False, prefix_sep='_'
    )

In [9]:
data

,label,Presence of Racial Slurs_No,Presence of Racial Slurs_Yes,Mention of Ethnic Groups_No,Mention of Ethnic Groups_Yes,Sentiment Polarity_Negative,Sentiment Polarity_Neutral,Sentiment Polarity_Positive,Use of Violent Language_No,Use of Violent Language_Yes,Presence of Hate Symbols_No,Presence of Hate Symbols_Yes,Use of Sarcasm_No,Use of Sarcasm_Yes,Text Length_Long,Text Length_Medium,Text Length_Short,Use of Expletives_No,Use of Expletives_Yes,Grammatical Errors_No,Grammatical Errors_Yes,Use of First-Person Pronouns_No,Use of First-Person Pronouns_Yes,Use of Second-Person Pronouns_No,Use of Second-Person Pronouns_Yes,Use of Third-Person Pronouns_No,Use of Third-Person Pronouns_Yes,Presence of Questions_No,Presence of Questions_Yes,Presence of Negations_No,Presence of Negations_Yes,Use of Hyperbolic Language_No,Use of Hyperbolic Language_Yes,Presence of Emotive Language_No,Presence of Emotive Language_Yes,Use of Metaphors_No,Use of Metaphors_Yes,Presence of Direct Quotes_No,Presence of Direct Quotes_Yes,Use of Capitalization for Emphasis_No,Use of Capitalization for Emphasis_Yes,Presence of Lists_No,Presence of Lists_Yes
0,0,True,False,True,False,False,True,False,True,False,True,False,True,False,False,True,False,True,False,True,False,True,False,True,False,False,True,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False
1,0,True,False,True,False,False,True,False,True,False,True,False,True,False,False,True,False,True,False,True,False,False,True,False,True,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False
2,0,True,False,True,False,False,True,False,True,False,True,False,True,False,False,False,True,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False
3,1,True,False,False,True,True,False,False,False,True,True,False,True,False,True,False,False,True,False,True,False,True,False,True,False,False,True,True,False,True,False,False,True,False,True,True,False,True,False,False,True,True,False
4,0,True,False,True,False,False,True,False,True,False,True,False,True,False,False,False,True,True,False,True,False,True,False,False,True,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6398,1,True,False,True,False,True,False,False,False,True,True,False,True,False,False,False,True,True,False,False,True,False,True,True,False,False,True,True,False,True,False,True,False,False,True,True,False,False,True,True,False,True,False
6399,0,True,False,True,False,True,False,False,True,False,True,False,False,True,False,False,True,True,False,True,False,True,False,True,False,False,True,True,False,False,True,True,False,False,True,True,False,True,False,True,False,True,False
6400,0,True,False,True,False,False,False,True,True,False,True,False,True,False,False,False,True,True,False,False,True,True,False,True,False,True,False,True,False,True,False,True,False,False,True,True,False,True,False,True,False,True,False
6401,0,True,False,True,False,False,False,True,True,False,True,False,True,False,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,True,False,False,True,True,False,True,False,True,False,True,False


In [10]:
# 1) Libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 2) Příprava feature matic X a cílové proměnné y
X = data.drop(columns=["label"])
y = data["label"]
#categorical_columns = X.select_dtypes(include=["object"]).columns
#X = df.drop(columns=categorical_columns)

# 3) Rozdělení na trénovací a testovací sadu
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)  # stratify, pokud je to klasifikace s nerovnoměrnými třídami

# ----------------------------------------------------------------
# 4) Definice modelu RandomForestClassifier
model = RandomForestClassifier(random_state=42, class_weight='balanced')

# 5) Nastavení rozsahu parametrů pro RandomizedSearchCV
param_grid = {
    "n_estimators": [50, 100, 200],       # Počet stromů v lese
    "max_depth": [3, 5, 10, None],        # Maximální hloubka stromu
    "min_samples_split": [2, 5, 10],      # Minimální počet vzorků pro split
    "min_samples_leaf": [1, 2, 5],        # Minimální počet vzorků v listu
}

# 6) Konfigurace RandomizedSearchCV (n_iter a cv lze upravit dle potřeby)
random_search = GridSearchCV(
    model,
    param_grid=param_grid,
    cv=5,                  # 5-fold cross-validace
    scoring="accuracy",    # metrika, dle které se bude model porovnávat
    n_jobs=-1,             # využití všech CPU jader pro rychlejší výpočet
    verbose=1
)

# 7) Trénink modelu s vyhledáváním nejlepších hyperparametrů
random_search.fit(X_train, y_train)

# 8) Vypsání nejlepších parametrů a skóre
print("Nejlepší parametry:", random_search.best_params_)
print("Nejlepší skóre na trénovací cross-validaci:", random_search.best_score_)

# 9) Ověření na testovací sadě
best_model = random_search.best_estimator_  # získáme nejlepší nalezený model
y_pred = best_model.predict(X_test)

# 10) Vyhodnocení
print("Přesnost na testu:", accuracy_score(y_test, y_pred))
print("Classification report na testu:")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Nejlepší parametry: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Nejlepší skóre na trénovací cross-validaci: 0.6845017149390243
Přesnost na testu: 0.6697892271662763
Classification report na testu:
              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1112
           1       0.37      0.45      0.40       146
           2       0.03      0.57      0.05         7
           3       0.02      0.12      0.04        16

    accuracy                           0.67      1281
   macro avg       0.33      0.46      0.32      1281
weighted avg       0.83      0.67      0.74      1281

